In [18]:
import pymysql
import pandas as pd
import json
import os
import sys
import requests
# sys.path.append("/home/trade_core")
from postgres_client import InitDBClient
from psycopg2 import extras
import time

In [4]:
with open('/home/trade_core/trade_core_config.json') as config_file:
    config = json.load(config_file)

node_db_name = config['node_settings'][config['node']]['db_settings']
node_db_setting_dict = config['database_setting'][node_db_name]

In [11]:
db_client = InitDBClient(**{**node_db_setting_dict, 'database': 'trade_core'})

InitDBClient|SCHEMA: trade_core already exists.


In [6]:
db_client.create_all_table()

InitDBClient|TABLE: user_info created.
InitDBClient|TABLE: exchange_config created.


In [8]:
def is_table_empty(conn, table_name):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cur.fetchone()[0]
        return count == 0

def get_column_names(conn, table_name):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position;
        """, (table_name,))
        return [row[0] for row in cur.fetchall()]

In [14]:
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
print(f"is table empty: {is_table_empty(conn, 'trade_config')}")
db_client.pool.putconn(conn)

is table empty: False


In [15]:
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
print(f"get_column_names: {get_column_names(conn, 'trade_config')}")
db_client.pool.putconn(conn)

get_column_names: ['id', 'uuid', 'acw_user_uuid', 'telegram_id', 'send_times', 'send_term', 'registered_datetime', 'service_datetime_end', 'target_market_code', 'origin_market_code', 'target_market_uid', 'origin_market_uid', 'target_market_referral_use', 'origin_market_referral_use', 'target_market_cross', 'target_market_leverage', 'origin_market_cross', 'origin_market_leverage', 'target_market_margin_call', 'origin_market_margin_call', 'target_market_safe_reverse', 'origin_market_safe_reverse', 'target_market_risk_threshold_p', 'origin_market_risk_threshold_p', 'repeat_limit_p', 'repeat_limit_direction', 'repeat_num_limit', 'on_off', 'remark']


In [32]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.18970322608947754


In [33]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.12562179565429688


In [43]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.*
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.service_datetime_end >= %s;"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql, val)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.3118107318878174


In [45]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    # sql = """
    #     SELECT trade.*, trade_config.*
    #     FROM trade
    #     JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.service_datetime_end >= %s;"""
    sql = """
        SELECT trade.*, trade_config.*
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.40634870529174805


In [51]:
result[0]

RealDictRow([('id', 2),
             ('uuid', '57f5df1d-95f9-49eb-b7e4-c15042c9b59f'),
             ('trade_config_uuid', '57f5df1d-95f9-49eb-b7e4-c15042c9b59f'),
             ('registered_datetime',
              datetime.datetime(2024, 2, 1, 9, 14, 47, 168124)),
             ('last_updated_datetime',
              datetime.datetime(2024, 2, 2, 8, 59, 21, 317123)),
             ('base_asset', 'BTC'),
             ('usdt_conversion', False),
             ('low', Decimal('3.000')),
             ('high', Decimal('4.000')),
             ('trigger_switch', None),
             ('trade_switch', None),
             ('trade_capital', None),
             ('enter_target_market_order_id', None),
             ('enter_origin_market_order_id', None),
             ('exit_target_market_order_id', None),
             ('exit_origin_market_order_id', None),
             ('status', None),
             ('remark', None),
             ('acw_user_uuid', 'd1b3f6e0-5b9e-4b0e-8b0a-9f6f5f5e0b1a'),
             ('

In [35]:
0.36/1000

0.00035999999999999997

In [ ]:
# First check whether the table is empty
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
if is_table_empty(conn, 'trade_config'):
    # # Get the column names
    # column_names = self.get_column_names(conn, table_name)
    # # Create empty dataframe
    # self.exchange_config_df = pd.DataFrame(columns=column_names)
    pass
else:
    curr.execute(f"SELECT * FROM {table_name}")
    exchange_config_df = pd.DataFrame(curr.fetchall())
    target_market_code_unique = exchange_config_df['target_market_code'].unique()
    origin_market_code_unique = exchange_config_df['origin_market_code'].unique()
    for target_market_code in target_market_code_unique:
        for origin_market_code in origin_market_code_unique:
            market_combi_code = f"{target_market_code}:{origin_market_code}"
            self.exchange_config_df_dict[market_combi_code] = exchange_config_df[(exchange_config_df['target_market_code'] == target_market_code) & (exchange_config_df['origin_market_code'] == origin_market_code)]
self.postgres_client.pool.putconn(conn)
if self.exchange_config_df_dict_initiated is False:
    self.exchange_config_df_dict_initiated = True

In [5]:
response = requests.get("https://arbicrypto.net/api/users/users/")